# Bag-of-n-Grams

_Feature Engineering_

**Count word-pairs and word-triples, not just single words — so "not good" can become its own feature.**

Bag-of-words turns a document into a vector of word counts and forgets the order. "dog bites man" and "man bites dog" get the exact same vector. That lost order sometimes carries the whole meaning — especially negation: "good" and "not good" are opposites, but bag-of-words sees the same word "good" in both.

       Bag-of-n-grams recovers a little of that order. Instead of counting only single words, it also counts contiguous runs of $n$ words. Now "not good" is its own feature, separate from "good" and "not". A sentiment model can learn that the "not good" column predicts a bad review.

---

This notebook is a practice scaffold for the **Bag-of-n-Grams** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Lesson: Bag-of-n-grams — count word PAIRS (bigrams), not just single words.
# Bag-of-words (unigrams) throws away word ORDER: it only counts how many times
# each word appears. So "good" and "not good" look almost the SAME to the model —
# both contain the word "good". A sentiment classifier then MISHANDLES NEGATION:
# it learns "good -> positive" and gets fooled by "not good".
# FIX: ngram_range=(1, 2) also counts adjacent word PAIRS like "not good" and
# "not bad". Now "not good" has its OWN feature, distinct from "good", so the
# SAME classifier can learn that "not good" is negative. Order is partly recovered.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

# ---- STEP 1: Load real data --------------------------------------------------
# A small REAL inline sentiment corpus (no downloads). Every example hinges on
# negation: the SAME adjective flips meaning when "not" is in front of it.
# label 1 = positive, 0 = negative.
# The corpus teaches BOTH that "good"/"bad" carry sentiment AND that "not good"/
# "not bad" flip it. The unigram model can only see the first lesson (it has no
# "not good" feature); the bigram model can see both. Held-out test = unseen
# SENTENCES that contain these negation phrases, so neither model memorized them.
train_texts = [
    # plain positives (label 1)
    "good", "this is good", "good film", "really good", "so good",
    # plain negatives (label 0)
    "bad", "this is bad", "bad film", "really bad", "so bad",
    # "not good" -> negative (label 0)
    "not good", "it was not good", "not good at all", "the acting is not good",
    # "not bad" -> positive (label 1)
    "not bad", "it was not bad", "not bad at all", "the acting is not bad",
]
train_labels = np.array(
    [1] * 5 +    # good ...
    [0] * 5 +    # bad ...
    [0] * 4 +    # not good ...
    [1] * 4      # not bad ...
)

# Held-out NEGATION sentences the model never saw verbatim. The UNIGRAM model
# only has "not"/"good"/"bad" columns, so it leans on "good"->positive and gets
# fooled; the BIGRAM model has a "not good"/"not bad" column and gets it right.
test_texts = [
    "the movie was not good",      # neg
    "honestly not good",            # neg
    "a not good experience",        # neg
    "the movie was not bad",        # pos
    "honestly not bad",             # pos
    "a not bad experience",         # pos
]
test_labels = np.array([0, 0, 0, 1, 1, 1])  # not good = neg, not bad = pos

# ---- STEP 2: Visualize the PURE (raw) representation -------------------------
# Vectorize with UNIGRAMS only (plain bag-of-words). Show the raw vectors for
# "good" vs "not good": they share the "good" column and differ only by "not".
uni_vec = CountVectorizer(ngram_range=(1, 1))
uni_vec.fit(train_texts + test_texts)
v_good_u = uni_vec.transform(["good"]).toarray()[0]
v_notgood_u = uni_vec.transform(["not good"]).toarray()[0]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
words = uni_vec.get_feature_names_out()
keep = np.where((v_good_u + v_notgood_u) > 0)[0]  # only columns either phrase uses
x = np.arange(len(keep))
ax[0].bar(x - 0.2, v_good_u[keep], 0.4, label='"good"', color="#27ae60")
ax[0].bar(x + 0.2, v_notgood_u[keep], 0.4, label='"not good"', color="#c0392b")
ax[0].set_xticks(x); ax[0].set_xticklabels(words[keep], rotation=0)
ax[0].set_ylabel("count")
ax[0].set_title("STEP 2: UNIGRAM bag-of-words\n'good' vs 'not good' — only differ by one 'not' column")
ax[0].legend()

# ---- STEP 3: Reproduce the PROBLEM on the raw (unigram) features -------------
# How similar do "good" and "not good" look to the model? Cosine similarity of
# their unigram vectors is HIGH (they share "good"), so they're easy to confuse.
cos_uni = cosine_similarity(v_good_u.reshape(1, -1), v_notgood_u.reshape(1, -1))[0, 0]
print(f'STEP 3  UNIGRAM cosine("good", "not good") = {cos_uni:.3f}  (high -> looks similar)')

# Train the SAME classifier on unigram features, score held-out negation examples.
X_train_u = uni_vec.transform(train_texts)
clf_u = LogisticRegression(max_iter=1000, random_state=0).fit(X_train_u, train_labels)
uni_acc = clf_u.score(uni_vec.transform(test_texts), test_labels)
print(f"STEP 3 PROBLEM  unigram bag-of-words, held-out negation accuracy = {uni_acc:.3f}")

# ---- STEP 4: Apply the technique (bigrams), visualize the engineered data ----
# ngram_range=(1, 2): keep unigrams AND adjacent pairs. Now "not good" gets its
# own column, so "good" and "not good" no longer share their non-zero features.
bi_vec = CountVectorizer(ngram_range=(1, 2))
bi_vec.fit(train_texts + test_texts)
v_good_b = bi_vec.transform(["good"]).toarray()[0]
v_notgood_b = bi_vec.transform(["not good"]).toarray()[0]

feats = bi_vec.get_feature_names_out()
keep_b = np.where((v_good_b + v_notgood_b) > 0)[0]
xb = np.arange(len(keep_b))
ax[1].bar(xb - 0.2, v_good_b[keep_b], 0.4, label='"good"', color="#27ae60")
ax[1].bar(xb + 0.2, v_notgood_b[keep_b], 0.4, label='"not good"', color="#c0392b")
ax[1].set_xticks(xb); ax[1].set_xticklabels(feats[keep_b], rotation=20, ha="right")
ax[1].set_ylabel("count")
cos_bi = cosine_similarity(v_good_b.reshape(1, -1), v_notgood_b.reshape(1, -1))[0, 0]
ax[1].set_title(f"STEP 4: with BIGRAMS (1,2)\n'not good' is its own feature — cosine drops to {cos_bi:.2f}")
ax[1].legend()
plt.tight_layout(); plt.show()

print(f'STEP 4  BIGRAM  cosine("good", "not good") = {cos_bi:.3f}  (lower -> now distinct)')

# ---- STEP 5: Show the FIX ----------------------------------------------------
# SAME LogisticRegression, now on (1,2)-gram features. It can finally tell that
# "not good" is negative, so held-out negation accuracy jumps.
X_train_b = bi_vec.transform(train_texts)
clf_b = LogisticRegression(max_iter=1000, random_state=0).fit(X_train_b, train_labels)
bi_acc = clf_b.score(bi_vec.transform(test_texts), test_labels)
print(f"STEP 5 FIX      bag-of-bigrams, held-out negation accuracy = {bi_acc:.3f}")

print(f"PROBLEM (unigram): {uni_acc:.3f}   ->   FIX (bigram): {bi_acc:.3f}")

## Reference implementation — scikit-learn (CountVectorizer), pandas

In [ ]:
# pip install scikit-learn pandas
# Dataset: Yelp Dataset Challenge (round 6) reviews, yelp.com/dataset
# Book code: github.com/alicezheng/feature-engineering-book (Chapter 3)
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# --- load the Yelp review text (one JSON object per line) ---
review_file = "yelp_academic_dataset_review.json"
reviews = []
with open(review_file) as f:
    for line in f:
        reviews.append(json.loads(line))
review_df = pd.DataFrame(reviews)
texts = review_df["text"]          # the raw review strings

# --- Bag-of-words: single words only (unigrams) ---
bow = CountVectorizer()            # default ngram_range=(1, 1)
bow.fit(texts)
print("unigram features :", len(bow.get_feature_names_out()))   # ~ 29,000

# --- Bag-of-bigrams: word-pairs only ---
bigram = CountVectorizer(ngram_range=(2, 2))
bigram.fit(texts)
print("bigram features  :", len(bigram.get_feature_names_out())) # ~ 357,000

# --- Bag-of-trigrams: word-triples only ---
trigram = CountVectorizer(ngram_range=(3, 3))
trigram.fit(texts)
print("trigram features :", len(trigram.get_feature_names_out())) # ~ 1,627,000

# --- The useful setting: unigrams AND bigrams together ---
uni_bi = CountVectorizer(ngram_range=(1, 2))
X = uni_bi.fit_transform(texts)    # sparse document x n-gram count matrix
print("uni+bigram feats :", len(uni_bi.get_feature_names_out()))

# negation now has its own feature: "not good" is distinct from "good"
vocab = uni_bi.vocabulary_
print('"good" index     :', vocab.get("good"))
print('"not good" index :', vocab.get("not good"))

# the cost is sparsity: a huge, mostly-zero matrix
print("matrix shape     :", X.shape, " stored nonzeros:", X.nnz)

# tame the explosion: keep only n-grams seen in many reviews
pruned = CountVectorizer(ngram_range=(1, 2), min_df=10, max_features=50000)
pruned.fit(texts)
print("pruned features  :", len(pruned.get_feature_names_out()))

## Visualize the data & results

_On a tiny real corpus of reviews (one with the negation "not good"), how fast does the number of distinct features grow as we widen ngram_range from (1,1) to (1,2) to (1,3)?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# six short REAL reviews; one carries a negation ("not good")
corpus = [
    "the food is good",
    "the food is not good",
    "great service and great food",
    "service was not great",
    "i would not come back",
    "good food and good service",
]

# count distinct features for three n-gram ranges
counts = {}
for rng in [(1, 1), (1, 2), (1, 3)]:
    vec = CountVectorizer(ngram_range=rng)
    vec.fit(corpus)
    counts[rng] = len(vec.get_feature_names_out())
    print("ngram_range", rng, "->", counts[rng], "features")
# ngram_range (1, 1) -> 12 features
# ngram_range (1, 2) -> 31 features
# ngram_range (1, 3) -> 45 features
# (sklearn's default token pattern drops 1-char tokens like "i")

# the (1,2) bag now has "not good" as its OWN feature, distinct from "good"
vec12 = CountVectorizer(ngram_range=(1, 2)).fit(corpus)
vocab = vec12.vocabulary_
print('"good" in vocab     :', "good" in vocab)        # True
print('"not good" in vocab :', "not good" in vocab)    # True  <- recovered negation

# the cost: a wider, sparser document x feature matrix
X = vec12.transform(corpus)
print("matrix shape", X.shape, "nonzeros", X.nnz,
      "sparsity %.1f%%" % (100 * (1 - X.nnz / (X.shape[0] * X.shape[1]))))

import matplotlib.pyplot as plt
ranges = ["(1,1)", "(1,2)", "(1,3)"]
vals = [counts[(1, 1)], counts[(1, 2)], counts[(1, 3)]]
plt.bar(ranges, vals, color=["#4ea1ff", "#7ee787", "#ffb454"])
plt.ylabel("number of distinct features")
plt.title("Feature count explodes as ngram_range widens")
for i, v in enumerate(vals):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The review i would not order this again has 6 words. How many bigrams ($n=2$) and trigrams ($n=3$) does it produce? Which bigram carries the negation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $L - n + 1$ with $L = 6$. Bigrams: $6 - 2 + 1 = 5$. Trigrams: $6 - 3 + 1 = 4$. — _Slide a window of width $n$ across the 6 words._
- List the bigrams: "i would", "would not", "not order", "order this", "this again". — _Each adjacent pair is one bigram feature._
- "not order" (and "would not") is the negation-bearing pair. — _Bag-of-words would split "not" from "order"; the bigram keeps the negation as one feature._

**Answer:** 5 bigrams and 4 trigrams. The bigram "not order" (helped by "would not") carries the negation that plain bag-of-words loses.

</details>

**Problem 2.** On Yelp the book reports about 29,000 unigrams, 357,000 bigrams, and 1,627,000 trigrams. Roughly how many times more features do bigrams and trigrams give versus unigrams, and what does that imply about the matrix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bigrams vs unigrams: $357\text{k} / 29\text{k} \approx 12$. Trigrams vs unigrams: $1{,}627\text{k} / 29\text{k} \approx 56$. — _Divide each count by the unigram count to get the multiplier._
- So adding bigrams makes the feature space ~12x wider; trigrams ~56x wider. — _Distinct n-grams grow far faster than the vocabulary._
- Each document still contains only a handful of those n-grams, so almost every column is zero. — _More columns with the same few hits per row means a much sparser matrix._

**Answer:** Bigrams ~12x and trigrams ~56x more features than unigrams. The matrix becomes far wider and much sparser — the cost you pay for the extra local-order signal. Prune with min_df / max_features.

</details>

**Problem 3.** A sentiment model on bag-of-words keeps calling "the food was not good" positive. Why, and what single change to the vectorizer most directly fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bag-of-words ($n=1$) has separate "not" and "good" features; the strong positive weight on "good" wins. — _Unigrams cannot represent that "not" negates "good" — order is discarded._
- Switch to a bag-of-n-grams that includes bigrams: CountVectorizer(ngram_range=(1,2)). — _This adds the "not good" feature, which the model can weight negatively._
- Optionally add min_df to prune the new rare bigrams and control the blow-up. — _Most added bigrams are rare noise; pruning keeps the matrix manageable._

**Answer:** Unigrams split "not" from "good", so the positive "good" weight dominates. Set ngram_range=(1,2) so "not good" becomes its own feature the model can penalize — and use min_df to tame the feature explosion.

</details>